# Alaga Alerta · GenAI — Pipeline RAG

Execute `python avaliacao/avaliar.py` a partir da raiz do repositório para gerar um placar novo com Ollama local. As células exibem a busca e as respostas armazenadas nessa avaliação.


In [1]:
import sys
import json
from pathlib import Path

root = Path.cwd()
if not (root / "avaliacao").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))

from rag import BaseConhecimento, LIMIAR_BUSCA

base = BaseConhecimento()
placar = json.loads((root / "avaliacao" / "placar.json").read_text(encoding="utf-8"))
print("Modelo avaliado:", placar["modelo"])
print("Trechos na base:", len(base.trechos))


C:\Users\gukaw\dev\projetos\alaga\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo avaliado: llama3.2:3b
Trechos na base: 14


## 1. Busca semântica (item 03)

In [2]:
for t in base.buscar('meu carro ficou preso na enchente, o que faço?'):
    print(f"{t['score']:.3f}  [{t['id']}] {t['fonte']}")

0.886  [c03] Corpo de Bombeiros — Resgate em enchentes
0.884  [c06] Secretaria de Saúde — Pós-enchente
0.857  [c14] Defesa Civil — Evacuação


In [3]:
testset = json.loads((root / "avaliacao" / "testset.json").read_text(encoding="utf-8"))
print("recall@1 =", round(base.recall_at_k(testset, 1), 3))
print("recall@3 =", round(base.recall_at_k(testset, 3), 3))
print("limiar de recusa =", LIMIAR_BUSCA)
for tipo in ("fundamentada", "recusa"):
    scores = [base.buscar(c["pergunta"], k=1)[0]["score"] for c in testset if c["tipo"] == tipo]
    print(f"{tipo}, faixa de cosseno top-1: {min(scores):.3f} a {max(scores):.3f}")


recall@1 = 1.0


recall@3 = 1.0
limiar de recusa = 0.84


fundamentada, faixa de cosseno top-1: 0.872 a 0.927
recusa, faixa de cosseno top-1: 0.786 a 0.839


## 2. As 3 configurações numa mesma pergunta (itens 04, 05)

In [4]:
for resultado in placar["respostas"]:
    if resultado["id"] == "t01":
        print("===", resultado["config"], "===")
        print(json.dumps(resultado["resposta"], ensure_ascii=False, indent=2))


=== sem_contexto ===
{
  "resposta": "Se seu carro ficar preso na água, é importante agir com calma e segurança. Primeiramente, verifique se você está em uma área segura e longe de correntes fortes de água. Em seguida, tente localizar o ponto mais baixo do veículo para evitar que ele seja arrastado pela água. Se possível, use um pedaço de madeira ou outro objeto sólido para criar uma ponte entre o carro e a terra. Se você não tiver acesso a um pedaço de madeira, pode tentar usar um saco de areia ou outro material absorvente para ajudar a manter o carro afundado. Lembre-se de que a segurança é a prioridade, então evite tentar sair do veículo se ele estiver completamente submerso. Em vez disso, espere por ajuda ou tente se comunicar com os serviços de emergência. Além disso, é importante verificar se há algum bloqueio ou obstáculo que possa estar impedindo o fluxo da água e causando o carro a ficar preso.",
  "fonte": null,
  "recusa": false
}
=== rag_simples ===
{
  "resposta": "Mantenh

In [5]:
for resultado in placar["respostas"]:
    if resultado["id"] == "t13" and resultado["config"] == "rag_agente":
        print(resultado["pergunta"])
        print(json.dumps(resultado["resposta"], ensure_ascii=False, indent=2))


Quem ganhou o Campeonato Brasileiro de 2025?
{
  "resposta": "Não consta na minha base de conhecimento.",
  "fonte": null,
  "recusa": true
}


## 3. Placar completo (itens 06 e 07)

In [6]:
print(json.dumps(placar["placar"], ensure_ascii=False, indent=2))
erros = json.loads((root / "avaliacao" / "erros.json").read_text(encoding="utf-8"))
print("Erros registrados:", len(erros))


[
  {
    "config": "sem_contexto",
    "acerto_conteudo": 0.3,
    "acerto_fundamentada": 0.0,
    "recusa_correta": 0.0,
    "taxa_respostas_divergentes": 0.786
  },
  {
    "config": "rag_simples",
    "acerto_conteudo": 0.9,
    "acerto_fundamentada": 0.3,
    "recusa_correta": 1.0,
    "taxa_respostas_divergentes": 0.071
  },
  {
    "config": "rag_agente",
    "acerto_conteudo": 0.9,
    "acerto_fundamentada": 0.7,
    "recusa_correta": 1.0,
    "taxa_respostas_divergentes": 0.071
  }
]
Erros registrados: 24
